In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS pep_dev.eval;

CREATE OR REPLACE TABLE pep_dev.eval.own_stmt AS 
SELECT entity_id, canonical_id, schema, prop, prop_type, dataset,value
FROM pep_dev.bronze.statements
WHERE _snapshot_version= (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
AND SCHEMA IN ('Ownership', 'Company', 'Organization', 'LegalEntity');


-- SELECT * FROM pep_dev.bronze.statements
-- LIMIT 10

In [0]:
-- SELECT * FROM pep_dev.eval.own_stmt;

SELECT prop , count(*) as n_rows, COUNT(DISTINCT entity_id) as n_entities,
ROUND (100*COUNT(DISTINCT entity_id)/t.total,1) as coverage_pct,t.total
FROM pep_dev.eval.own_stmt
CROSS JOIN (SELECT COUNT(DISTINCT entity_id) as total FROM pep_dev.eval.own_stmt
WHERE schema='Ownership') t 
WHERE SCHEMA ='Ownership'
GROUP BY prop,t.total
ORDER BY n_entities DESC


In [0]:
SELECT * FROM pep_dev.eval.own_stmt
LIMIT 100;

In [0]:
%sql

SELECT VALUE,count(*) FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage'
 
GROUP BY value

ORDER BY value DESC

LIMIT 10000

In [0]:
%sql

SELECT try_cast(VALUE AS double),value,count(*) FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage' AND try_cast(VALUE AS double) IS NULL 
 
GROUP BY value

ORDER BY value DESC

-- LIMIT 10000

In [0]:
%sql

SELECT 
    count_if(try_cast(value AS DOUBLE) = 0) as zero_pct,
    count_if(try_cast(value AS DOUBLE) < 1) as below_one,
    count_if(try_cast(value AS DOUBLE) > 100) as above_100

  FROM pep_dev.eval.own_stmt

WHERE prop = 'percentage' AND schema IN ('Ownership') 

-- HAVING count_if(try_cast(value AS DOUBLE) IS MAX() )
 


In [0]:
    SELECT
        dataset,
        COUNT(*)                     AS n_rows,
        MIN(try_cast(value AS DOUBLE))      AS min_pct,
        MAX(try_cast(value AS DOUBLE)  )      AS max_pct,
        count_if( try_cast(value AS DOUBLE) > 0 AND try_cast(value AS DOUBLE) < 1 )  AS below_one
    FROM   pep_dev.eval.own_stmt
    WHERE  prop = 'percentage' AND schema IN ('Ownership') 
    GROUP BY dataset
    ORDER BY max_pct 